In [ ]:
import torch
from torch import optim, nn
import torchvision
from torchvision import datasets, models, transforms
import numpy as np
from torch.utils.data.sampler import SubsetRandomSampler
import os
import pandas as pd
from PIL import Image
import albumentations as A


In [ ]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
def get_model(device):
  model = models.resnet18(weights='IMAGENET1K_V1') 

  hidd_size = model.fc.in_features // 3
  model.fc = nn.Sequential(nn.Linear(model.fc.in_features, hidd_size), nn.ReLU(), nn.Linear(hidd_size, 2),
                           nn.LogSoftmax(dim=1)) 

    
  for name, child in model.named_children():
    if name in ['layer3', 'layer4', 'fc']:
      print(name + ' is unfrozen')
      for param in child.parameters():
        param.requires_grad = True
    else:
      print(name + ' is frozen')
      for param in child.parameters():
        param.requires_grad = False

  model.to(device)
  return model

In [ ]:
model = get_model(device)

In [ ]:
checkpoint = torch.load("model.pt", map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

In [ ]:
from PIL import Image
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
classes = {0: "phone", 1: "remote"}

image = Image.open("test.jpg").convert("RGB")
image = transform(image).unsqueeze(0).to(device)

with torch.no_grad():
    logps = model(image)
    pred = logps.argmax(dim=1).item()

print("Предсказание:", classes[pred])


In [ ]:
image = Image.open("test2.png").convert("RGB")
image = transform(image).unsqueeze(0).to(device)

with torch.no_grad():
    logps = model(image)
    pred = logps.argmax(dim=1).item()

print("Предсказание:", classes[pred])
